Import simulation outputs from LISFLOOD-FP and package as NetCDF files.

Ensure to specify `voutput` in LISFLOOD-FP parameters file to get velocity as well as water depth.

In [1]:
import os
import rioxarray as rxr
from typing import cast, Literal
import xarray

LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/mSWE-GNN-train-test"
PREFIX = "res_5m_acc_cuda"
MAX_STEP = 40
os.chdir(LISFLOOD_OUTPUT_DIR)

def read_step(step: int, prefix: str, ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"]|None=None) -> xarray.DataArray:
        return cast(xarray.DataArray, rxr.open_rasterio(f"{prefix}-{int(step):04}.{ftype}", parse_coordinates=True, masked=True))[0]

In [7]:
def extract_parameter(ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"], max_step, prefix=PREFIX, shape=None):
    results = []
    for step in range(0, max_step+1):
        results.append(read_step(step, prefix, ftype))
    parameter_array = xarray.DataArray(results, dims=["time","x","y"])
    print(parameter_array)
    if shape:
        parameter_array = parameter_array.sel(x = slice(0, shape[0]), y = slice(0, shape[1]))
    return parameter_array

In [8]:
wd = extract_parameter("wd", MAX_STEP)
Vx = extract_parameter("Vx", MAX_STEP, shape=wd.shape[1:])
Vy = extract_parameter("Vy", MAX_STEP, shape=wd.shape[1:])

simulation_output = xarray.Dataset({
    "mesh2d_waterdepth": wd.stack(mesh2d_nFaces=("x","y")),
    "mesh2d_ucx": Vx.stack(mesh2d_nFaces=("x","y")),
    "mesh2d_ucy": Vy.stack(mesh2d_nFaces=("x","y"))
})

<xarray.DataArray (time: 41, x: 851, y: 356)> Size: 50MB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
...
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..

In [10]:
simulation_output.reset_index("mesh2d_nFaces").to_netcdf("dyce_0.nc", format="NETCDF4")